In [6]:
import os
import mne
import pyedflib
import numpy as np
import scipy
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import jupyterlab
import yasa
import glob
import gc
import warnings

# Configuration for interactive plots
%matplotlib widget

# Safe import of psutil with error handling in case it is not installed
try:    
    import psutil
except ImportError:    
    psutil = None

# Definition of the RAM check function
def print_ram(label=""):
    if psutil is None:
        print(f"{label} RAM: unavailable (psutil not installed)")
        return
        
    process = psutil.Process(os.getpid())
    mb = process.memory_info().rss / 1024 / 1024
    print(f"{label} RAM: {mb:.1f} MB")


print_ram("Initial state")

Initial state RAM: 89.9 MB


In [7]:
# Plot and save the clean PSD


In [8]:

# Path to the folder containing your .edf files (change to your actual path)
data_folder = "/Users/alicjakawecka/Desktop/CZD_sampleData/CZD_sampleData/eeg_edf"

# Retrieve a list of paths to all .edf files in that folder
edf_files = glob.glob(f"{data_folder}/*.edf")

# Display the found files and their total count
print(f"Found {len(edf_files)} EDF files.")
for file in edf_files:
    print(file)

Found 4 EDF files.
/Users/alicjakawecka/Desktop/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-26{6591E658-D0E0-49FE-BDE4-FC101EACF45D} Data.edf
/Users/alicjakawecka/Desktop/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-27{891F8634-3CC8-4FEE-B1D3-64394DC86F7A} Data.edf
/Users/alicjakawecka/Desktop/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-27{0138DAF9-E55E-4A8F-9718-71DB42B18BDB} Data.edf
/Users/alicjakawecka/Desktop/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-27{D29553B9-1220-4DFC-8456-F712EE2281C8} Data.edf


In [9]:
def create_bipolar(raw_input):

    electrodes = {}

    for ch in raw_input.ch_names:
        if not ch[-1].isdigit():
            continue

        electrode = ch.rstrip("0123456789")

        if electrode not in electrodes:
            electrodes[electrode] = []

        electrodes[electrode].append(ch)

    anode = []
    cathode = []
    bipolar_names = []

    for electrode in electrodes:
        # Sort channels by their actual number (integer value)
        channels = sorted(
            electrodes[electrode],
            key=lambda x: int(x[len(electrode):])
        )

        for i in range(len(channels) - 1):
            # Extract numerical numbers for the current and next channel
            current_num = int(channels[i][len(electrode):])
            next_num = int(channels[i + 1][len(electrode):])
            
            # CONDITION: Create a pair only if they are adjacent (the difference is exactly 1)
            if next_num - current_num == 1:
                anode.append(channels[i])
                cathode.append(channels[i + 1])
                bipolar_names.append(f"{channels[i]}-{channels[i + 1]}")
            else:
                # Optional: you can add a print statement here, e.g., print(f"Skipped gap: {channels[i]} to {channels[i+1]}")
                continue

    raw_bipolar = mne.set_bipolar_reference(
        raw_input,
        anode=anode,
        cathode=cathode,
        ch_name=bipolar_names,
        drop_refs=False,
        copy=True,
        verbose=False
    )
 
    return raw_bipolar, bipolar_names

In [11]:
# Loop over EDF files
for edf_file in edf_files:
    file_base_name = os.path.splitext(os.path.basename(edf_file))[0]
    
    # Step 1: Very clear print statement about starting the file
    print("\n" + "="*130)
    print(f'PROCESSING THIS FILE: {file_base_name} .......................')
    print("="*130)
    
    output_dir = f"SOZ/{file_base_name}"
    os.makedirs(output_dir, exist_ok=True)
    
    # Load metadata silently
    raw = mne.io.read_raw_edf(edf_file, preload=False, verbose=False)

    # Extract annotations
    annotations = pd.DataFrame({
        "onset_s": raw.annotations.onset,
        "duration_s": raw.annotations.duration,
        "description": raw.annotations.description
    })

    # Save the table and print info to the console
    annotations_path = f"{output_dir}/inventory_table_annotations.csv"
    annotations.to_csv(annotations_path, index=False)
    print(f"[INFO] Saved full inventory table to: {annotations_path}")

    # Search for seizures
    seizure_rows = annotations[annotations["description"].str.contains("NAPAD", na=False)]

    if seizure_rows.empty:
        print(f"[RESULT] No seizure ('NAPAD') found in this file. Skipping.")
        continue  

    # Print the total number of found seizures
    total_seizures = len(seizure_rows)
    print(f"[RESULT] Found {total_seizures} seizure event(s) ('NAPAD') in this file.")

    # Loop over seizures
    for index, row in enumerate(seizure_rows.itertuples(), start=1):
        onset = row.onset_s
        tmin = onset - 10
        tmax = onset + 30

        seizure_output_dir = f"{output_dir}/seizure_{index}"
        os.makedirs(seizure_output_dir, exist_ok=True)
        os.makedirs(f"{seizure_output_dir}/spectrograms", exist_ok=True)

        # Silent data cropping
        raw_cropped = raw.copy().crop(tmin=tmin, tmax=tmax, verbose=False)
        raw_cropped.load_data(verbose=False)

        # Silent filtering
        raw_filtered = raw_cropped.copy()
        raw_filtered.notch_filter(freqs=[50, 100, 150], fir_design='firwin', verbose=False)
        raw_filtered.filter(l_freq=1.0, h_freq=70.0, fir_design='firwin', verbose=False)

        # Bipolar montage
        raw_bipolar_filtered, bipolar_names = create_bipolar(raw_filtered)
        total_channels = len(bipolar_names)

        # Print info about filtered bipolar channels for this seizure
        print(f"  -> Seizure #{index}/{total_seizures} | Onset: {onset:.1f}s | Window: {tmin:.1f}s to {tmax:.1f}s | Channels filtered: {total_channels}")

        # Silent trace plotting
        fig_filtered = raw_bipolar_filtered.plot(
            picks=bipolar_names, n_channels=15, duration=10.0, scalings="auto",
            title=f"FILTERED Bipolar trace - Seizure {index}", show=False, verbose=False
        )
       #fig_filtered.savefig(f"{seizure_output_dir}/bipolar_filtered_trace.png", dpi=300)
        plt.close(fig_filtered)

        # 8. Compute PSD
        print(f"    Computing PSD for Seizure #{index}...")
        psd = raw_bipolar_filtered.compute_psd(
            fmin=0,
            fmax=40,
            picks=bipolar_names,
            verbose='ERROR'
        )

        # --- AUTOMATIC DEAD CHANNEL DETECTION ---
        # Get the underlying data matrix from the PSD object
        # Shape is usually (n_channels, n_frequencies)
        psd_data = psd.get_data()
        
        # Calculate the mean power across frequencies for each channel
        mean_power = np.mean(psd_data, axis=1)
        
        # Dead/flat channels will have a mean power close to 0 (or less than 1e-25 in absolute voltage scale)
        dead_indices = np.where(mean_power < 1e-20)[0]
        
        if len(dead_indices) > 0:
            dead_channels = [bipolar_names[i] for i in dead_indices]
            print(f"    [WARNING] Detected {len(dead_channels)} disconnected/flat channel(s): {', '.join(dead_channels)}")
            
            # Filter them out: Keep only the channels that are NOT dead
            bipolar_names = [ch for ch in bipolar_names if ch not in dead_channels]
            total_channels = len(bipolar_names)
            
            # Re-compute PSD without the dead channels so the plot looks clean
            psd = raw_bipolar_filtered.compute_psd(
                fmin=0,
                fmax=40,
                picks=bipolar_names,
                verbose='ERROR'
            )
        # ----------------------------------------
        with warnings.catch_warnings():
            # This suppresses the 
            # "/tmp/ipykernel_21482/2753428352.py:111: RuntimeWarning: Channel locations not available. Disabling spatial colors."
            #  warning just for this plot
            warnings.filterwarnings("ignore", category=RuntimeWarning, message=".*Channel locations not available.*")
            
        # Plot and save the clean PSD
            fig_psd = psd.plot(average=False, show=False)
            fig_psd.suptitle(
                f"PSD - bipolar channels - Seizure {index} (Cleaned)",
                fontsize=14
            )
            fig_psd.savefig(
                f"{seizure_output_dir}/psd_bipolar.png",
                dpi=300,
                bbox_inches="tight"    
            )   
            plt.close(fig_psd)
        
        print(f"     [SUCCESS] Saved 40s window trace and clean PSD graph.")
        """
        # Spectrogram generation (clean scipy/matplotlib, no logs)
        sfreq = raw_bipolar_filtered.info["sfreq"]
        for channel_for_spectrogram in bipolar_names:
            signal = raw_bipolar_filtered.get_data(picks=[channel_for_spectrogram], verbose=False)[0]
            f, t, Sxx = scipy.signal.spectrogram(signal, fs=sfreq, nperseg=512, noverlap=256)

            fig_spec = plt.figure(figsize=(10, 5))
            plt.pcolormesh(t, f, 10 * np.log10(Sxx), shading="gouraud")
            plt.ylim(0, 100)
            plt.xlabel("Time [s]")
            plt.ylabel("Frequency [Hz]")
            plt.title(f"Spectrogram - {channel_for_spectrogram} (Seizure {index})")
            plt.colorbar(label="Power [dB]")
            plt.tight_layout()

            fig_spec.savefig(f"{seizure_output_dir}/spectrograms/spectrogram_{channel_for_spectrogram}.png", dpi=300)
            plt.close(fig_spec)
        """
        # Memory cleanup
        del raw_cropped, raw_filtered, raw_bipolar_filtered, psd
        plt.close('all')
        gc.collect()

    print(f"\n[DONE] Finished processing file: {file_base_name}")


PROCESSING THIS FILE: 2023-07-26{6591E658-D0E0-49FE-BDE4-FC101EACF45D} Data .......................
[INFO] Saved full inventory table to: SOZ/2023-07-26{6591E658-D0E0-49FE-BDE4-FC101EACF45D} Data/inventory_table_annotations.csv
[RESULT] Found 6 seizure event(s) ('NAPAD') in this file.
  -> Seizure #1/6 | Onset: 15886.7s | Window: 15876.7s to 15916.7s | Channels filtered: 103


TypeError: 'fig' must be an instance of matplotlib.figure.Figure, int, str or None, not a mne_qt_browser._pg_figure.MNEQtBrowser